In [7]:
#imports
import os
import random
import re
import pandas as pd
import torch

#loading dataset
TRAIN_PATH = "/content/trac2_CONVT_train.csv"
DEV_PATH   = "/content/trac2_CONVT_dev.csv"
TEST_PATH  = "/content/goldstandard_CONVT_test.csv"

#model
MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"
MAX_NEW_TOKENS = 300
TEMPERATURE = 0.7
TOP_P = 0.9
SEED = 42
EVAL_SAMPLE_SIZE = 30

DEV_OUT_CSV = "/content/generations_icl_dev.csv"
TEST_OUT_CSV = "/content/generations_icl_test.csv"

random.seed(SEED)
torch.manual_seed(SEED)

#all the utility functions
def load_csv_safe(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing file: {path}")
    return pd.read_csv(path, on_bad_lines='skip')

def find_text_col(df):
    candidates = [c for c in df.columns if c.lower() == 'text']
    if candidates:
        return candidates[0]
    # fallback
    for c in df.columns:
        if df[c].dtype == object:
            return c
    raise ValueError("No text-like column found.")

# building histories and gold
def build_conv_maps(df):
    text_col = find_text_col(df)

    conv_map = {}
    for _, row in df.iterrows():
        cid = row["conversation_id"]
        tid = int(row["turn_id"])
        utt = str(row[text_col]) if pd.notna(row[text_col]) else ""
        conv_map.setdefault(cid, []).append((tid, utt))

    histories, golds = {}, {}

    for cid, turns in conv_map.items():
        turns_sorted = sorted(turns, key=lambda x: x[0])

        # First 5 = history
        history = [utt for _, utt in turns_sorted[:5]]

        # Next up to 10 = gold
        gold = [utt for _, utt in turns_sorted[5:15]]
        if len(gold) < 10:
            gold += [""] * (10 - len(gold))

        histories[cid] = history
        golds[cid] = gold

    return histories, golds

#loading all datasets
print("Loading datasets...")
train_df = load_csv_safe(TRAIN_PATH)
dev_df   = load_csv_safe(DEV_PATH)
test_df  = load_csv_safe(TEST_PATH)

print("Train:", train_df.shape)
print("Dev:", dev_df.shape)
print("Test:", test_df.shape)

print("Building conversation structures...")
train_histories, train_gold = build_conv_maps(train_df)
dev_histories, dev_gold     = build_conv_maps(dev_df)
test_histories, test_gold   = build_conv_maps(test_df)

print("Num conversations → train:", len(train_histories),
      "dev:", len(dev_histories),
      "test:", len(test_histories))

#few-shot prompt builders
def format_example_block(cid, history, gold):
    block = "### Example Conversation\n"
    block += f"Conversation ID: {cid}\n"
    block += "Input (Turns 1–5):\n"
    for i, utt in enumerate(history, start=1):
        block += f"Turn {i}: {utt}\n"
    block += "\nGold Output (Turns 6–15):\n"
    for i, utt in enumerate(gold, start=6):
        block += f"Turn {i}: {utt}\n"
    block += "\n"
    return block

def build_few_shot_prompt(k, dev_histories, dev_gold):
    # need to  have at least one non-empty gold turn
    candidates = [cid for cid, g in dev_gold.items() if any(x.strip() for x in g)]

    if len(candidates) < k:
        print(f"Only {len(candidates)} usable dev examples. Cannot build {k}-shot.")
        return None, []

    random.shuffle(candidates)
    selected = candidates[:k]

    prompt = f"### {k}-Shot Few-Shot Prompt for Dialogue Generation\n\n"
    for cid in selected:
        prompt += format_example_block(cid, dev_histories[cid], dev_gold[cid])
    prompt += "### End of Few-Shot Examples\n\n"

    return prompt, selected

fewshot_3_prompt, fs3_ids = build_few_shot_prompt(3, dev_histories, dev_gold)
fewshot_5_prompt, fs5_ids = build_few_shot_prompt(5, dev_histories, dev_gold)

if fewshot_5_prompt:
    BEST_PROMPT = fewshot_5_prompt
    print("Using 5-shot prompt.")
elif fewshot_3_prompt:
    BEST_PROMPT = fewshot_3_prompt
    print("Using 3-shot prompt.")
else:
    raise RuntimeError("No valid few-shot examples available.")

#loading LLM model
print("Loading model:", MODEL_NAME)

from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

if torch.cuda.is_available():
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,
        device_map="auto"
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, device_map={"": "cpu"}
    )

model.eval()
print("Model device:", model.device)

#prompting , generating and parsing
def build_generation_prompt(history_turns, fewshot_prompt):
    prompt = fewshot_prompt
    prompt += "### New Conversation\n"
    prompt += "Input (Turns 1–5):\n"
    for i, utt in enumerate(history_turns, start=1):
        prompt += f"Turn {i}: {utt}\n"
    prompt += """
Your task: Generate the next 10 turns (Turn 6 to Turn 15).
Format strictly as:

Turn 6: ...
Turn 7: ...
Turn 8: ...
Turn 9: ...
Turn 10: ...
Turn 11: ...
Turn 12: ...
Turn 13: ...
Turn 14: ...
Turn 15: ...
"""
    return prompt

def parse_generated_turns(decoded, full_prompt):
    if decoded.startswith(full_prompt):
        gen = decoded[len(full_prompt):]
    else:
        gen = decoded

    pattern = r"Turn\s+([6-9]|1[0-5])\s*:\s*(.+?)(?=(?:Turn\s+[6-9]|Turn\s+1[0-5]|$))"
    matches = re.findall(pattern, gen, flags=re.DOTALL)

    parsed = [t.strip() for _, t in matches]

    if len(parsed) < 10:
        parsed += [""] * (10 - len(parsed))
    elif len(parsed) > 10:
        parsed = parsed[:10]

    return parsed

def generate_next_10_turns(history, fewshot_prompt):
    full_prompt = build_generation_prompt(history, fewshot_prompt)
    inputs = tokenizer(full_prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        do_sample=True,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id
    )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    parsed = parse_generated_turns(decoded, full_prompt)
    return parsed

#evaluating on dev samples
print("Selecting evaluation subset of size", EVAL_SAMPLE_SIZE)
all_dev_ids = list(dev_histories.keys())
sample_cids = random.sample(all_dev_ids, min(EVAL_SAMPLE_SIZE, len(all_dev_ids)))
print("Eval IDs:", sample_cids)

import evaluate
rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")
bertscore = evaluate.load("bertscore")

def evaluate_prompt(prompt, label):
    print(f"\n=== Evaluating {label} ===")
    preds_all = []
    refs_all = []

    for cid in sample_cids:
        history = dev_histories[cid]
        gold = dev_gold[cid]
        preds = generate_next_10_turns(history, prompt)

        preds_all.extend(preds)
        refs_all.extend(gold)

    r = rouge.compute(predictions=preds_all, references=refs_all)
    b = bleu.compute(predictions=preds_all, references=refs_all)
    bs = bertscore.compute(predictions=preds_all, references=refs_all, lang="en")
    bs_f1 = sum(bs["f1"]) / len(bs["f1"])

    print("ROUGE:", r)
    print("BLEU:", b)
    print("BERTScore F1:", bs_f1)
    return r, b, bs_f1

if fewshot_3_prompt:
    evaluate_prompt(fewshot_3_prompt, "3-SHOT")

if fewshot_5_prompt:
    evaluate_prompt(fewshot_5_prompt, "5-SHOT")

#dev generation
print("\nGenerating full DEV predictions...")
dev_outputs = []

for cid, history in dev_histories.items():
    preds = generate_next_10_turns(history, BEST_PROMPT)
    for i, utt in enumerate(preds, start=6):
        dev_outputs.append({"id": cid, "turn_id": i, "generated_response": utt})

pd.DataFrame(dev_outputs).to_csv(DEV_OUT_CSV, index=False)
print("Saved DEV predictions →", DEV_OUT_CSV)

#test generation
print("\nGenerating full TEST predictions...")
test_outputs = []

for cid, history in test_histories.items():
    preds = generate_next_10_turns(history, BEST_PROMPT)
    for i, utt in enumerate(preds, start=6):
        test_outputs.append({"id": cid, "turn_id": i, "generated_response": utt})

pd.DataFrame(test_outputs).to_csv(TEST_OUT_CSV, index=False)
print("Saved TEST predictions →", TEST_OUT_CSV)

print("\nFinished Q2 Pipeline Successfully!")

Loading datasets...
Train: (11090, 12)
Dev: (987, 13)
Test: (2316, 10)
Building conversation structures...
Num conversations → train: 487 dev: 33 test: 67
Using 5-shot prompt.
Loading model: microsoft/Phi-3-mini-4k-instruct


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model device: cuda:0
Selecting evaluation subset of size 30
Eval IDs: [110, 98, 108, 101, 190, 150, 173, 107, 96, 172, 170, 184, 179, 74, 171, 121, 99, 144, 118, 163, 85, 105, 174, 80, 113, 68, 72, 175, 145, 102]



=== Evaluating 3-SHOT ===


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ROUGE: {'rouge1': np.float64(0.08606987078064014), 'rouge2': np.float64(0.004334655692050245), 'rougeL': np.float64(0.06662298858434265), 'rougeLsum': np.float64(0.06653176300007657)}
BLEU: {'bleu': 0.0, 'precisions': [0.1436073520500909, 0.004877014418999152, 0.0006693440428380187, 0.0], 'brevity_penalty': 0.6438335249915181, 'length_ratio': 0.6942925255924836, 'translation_length': 4951, 'reference_length': 7131}
BERTScore F1: 0.6664728490511577

=== Evaluating 5-SHOT ===


ROUGE: {'rouge1': np.float64(0.06519518015589336), 'rouge2': np.float64(0.004654779824605658), 'rougeL': np.float64(0.05239662881258246), 'rougeLsum': np.float64(0.05259939363327362)}
BLEU: {'bleu': 0.0, 'precisions': [0.13786112519006588, 0.005579171094580234, 0.00027909572983533354, 0.0], 'brevity_penalty': 0.44612929162103687, 'length_ratio': 0.55335857523489, 'translation_length': 3946, 'reference_length': 7131}
BERTScore F1: 0.5162924321492514

Generating full DEV predictions...
Saved DEV predictions → /content/generations_icl_dev.csv

Generating full TEST predictions...
Saved TEST predictions → /content/generations_icl_test.csv

🎉 Finished Q2 Pipeline Successfully!
